# Run direct MILP and Benders

Use this notebook after running both data-generation notebooks. It calls the existing single-commodity and multi-commodity solvers on the same instance files, including the current special-product routing restrictions.

**Start:** restart the kernel, review the settings below, then Run All. Defaults run **all 120 single-commodity and 120 multi-commodity instances with both algorithms: 480 solver runs**. Set `MAX_SCENARIOS = 1` for a smaller smoke test. Use `MODEL = "single"` or `"multi"` for just one model, or `ALGORITHM = "milp"` / `"benders"` for just one algorithm.

The existing `run_model_tests.ipynb` remains available for direct-MILP Excel exports, solution payloads, and plots. This notebook saves solver and comparison CSVs, a progress summary, run metadata, and Benders histories in a new timestamped results folder.

Both algorithms use **penalised overflow**. The current Benders API has an iteration limit but no wall-clock time limit; one master or subproblem solve may take a long time. Direct MILP is unlimited by default too. Gurobi thread and seed settings are left at their defaults in both algorithms. These are correctness diagnostics, not a fully controlled performance experiment.

In [ ]:
# Settings: edit this cell, then run the cells below in order.
MODEL = "both"             # "single", "multi", or "both"
ALGORITHM = "milp"         # "milp", "benders", or "both"
MAX_SCENARIOS = None       # All instances per model; set 1 for a smoke test.

# None uses the generated singlecommodity_scenarios / multicommodity_scenarios folders.
# For custom inputs, use a dictionary with separate folders or JSON files:
# SCENARIO_INPUTS = {"single": ["path/to/single"], "multi": ["path/to/multi"]}
SCENARIO_INPUTS = None

DIRECT_TIME_LIMIT_SECONDS = None  # Optional MILP-only limit; Benders remains unlimited in time.
BENDERS_MAX_ITERS = 200
BENDERS_TOL = 1e-4                # Relative Benders stopping tolerance.
COMPARISON_RTOL = 2e-4            # Objective agreement tolerance, allowing both solver gaps.
VERBOSE_BENDERS = False
STOP_ON_ERROR = False             # Record per-solve errors and continue through the full suite.
RESULTS_ROOT = "results/milp_benders"

## Setup and environment

In [2]:
import csv
import hashlib
import importlib
import json
import math
import os
import platform
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents)
                     if (p / "benders_multicommodity.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook from the repository or one of its subfolders")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Refuse a different interpreter/package set before generating or solving.
from scripts.check_environment import check_environment
environment_report = check_environment(check_solver=True)
if not environment_report["passed"]:
    raise RuntimeError(
        "Use MATH4003 requirements (Python 3.9.6, Gurobi 12.0.3) from .venv with an active Gurobi license. "
        + "; ".join(environment_report["errors"])
    )


import gurobipy as gp
import model_data_loader
import multicommodity_milp
import monolithic_milp
import benders_monolithic
import benders_multicommodity
import scenario_runner
for module in (model_data_loader, monolithic_milp, multicommodity_milp,
               benders_monolithic, benders_multicommodity, scenario_runner):
    importlib.reload(module)

assert MODEL in ("single", "multi", "both")
assert ALGORITHM in ("milp", "benders", "both")
assert MAX_SCENARIOS is None or (isinstance(MAX_SCENARIOS, int) and MAX_SCENARIOS > 0)
assert DIRECT_TIME_LIMIT_SECONDS is None or DIRECT_TIME_LIMIT_SECONDS > 0
assert BENDERS_MAX_ITERS > 0 and BENDERS_TOL > 0 and COMPARISON_RTOL > 0
SELECTED_MODELS = ["single", "multi"] if MODEL == "both" else [MODEL]
SELECTED_ALGORITHMS = ["milp", "benders"] if ALGORITHM == "both" else [ALGORITHM]
MODULES = {
    "single": (monolithic_milp, benders_monolithic),
    "multi": (multicommodity_milp, benders_multicommodity),
}
DEFAULT_INPUTS = {
    "single": PROJECT_ROOT / "synthetic_data_generator" / "singlecommodity_scenarios",
    "multi": PROJECT_ROOT / "synthetic_data_generator" / "multicommodity_scenarios",
}
print("Project:", PROJECT_ROOT)
print("Python:", platform.python_version(), "Gurobi:", gp.gurobi.version())
print("Models:", SELECTED_MODELS, "Algorithms:", SELECTED_ALGORITHMS)
print("Benders iteration cap:", BENDERS_MAX_ITERS, "(no wall-clock cap)")

Project: C:\Ponpot_Work_Space\math4003-port-yard-optimisation
Python: 3.9.6 Gurobi: (12, 0, 3)
Models: ['single', 'multi'] Algorithms: ['milp', 'benders']
Benders iteration cap: 200 (no wall-clock cap)


## Select and validate input files

Default generated folders are checked against their benchmark manifests, including the generator version. Every selected file must pass canonical validation before solving. If loading fails, regenerate the instances using the updated generation notebooks. The printed counts should be 120 per model for a full run.

In [3]:
def selected_paths(key):
    value = DEFAULT_INPUTS[key] if SCENARIO_INPUTS is None else SCENARIO_INPUTS[key]
    inputs = value if isinstance(value, (list, tuple)) else [value]
    resolved = [Path(value) if Path(value).is_absolute() else PROJECT_ROOT / value for value in inputs]
    paths = scenario_runner.scenario_paths([str(p) for p in resolved])
    if not paths:
        raise FileNotFoundError(f"No scenario JSON files found for {key}: {resolved}")
    if SCENARIO_INPUTS is None:
        manifest_path = DEFAULT_INPUTS[key] / "benchmark_manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        expected = {(DEFAULT_INPUTS[key] / entry["file"]).resolve() for entry in manifest["files"]}
        if {Path(path).resolve() for path in paths} != expected:
            raise ValueError(f"{key}: files differ from benchmark manifest; check stale or missing instances")
        # Preserve generation order so a one-instance run starts with F1-L.
        paths = [str((DEFAULT_INPUTS[key] / entry["file"]).resolve()) for entry in manifest["files"]]
        if manifest["instance_count"] != len(expected):
            raise ValueError(f"{key}: incorrect manifest instance count")
    else:
        manifest = None
    print(f"{key}: {len(paths)} available")
    chosen = paths if MAX_SCENARIOS is None else paths[:MAX_SCENARIOS]
    direct_module, _ = MODULES[key]
    for path in chosen:
        data = direct_module.load_scenario(path)
        if manifest is not None:
            version = data["raw"]["metadata"]["generator_randomization_version"]
            if version != manifest["generator_randomization_version"]:
                raise ValueError(f"Mixed generator versions: {path}")
    print(f"{key}: {len(chosen)} selected; first: {chosen[0]}")
    return chosen

PATHS_BY_MODEL = {key: selected_paths(key) for key in SELECTED_MODELS}
PLANNED_INSTANCES = sum(len(paths) for paths in PATHS_BY_MODEL.values())
PLANNED_SOLVER_RUNS = PLANNED_INSTANCES * len(SELECTED_ALGORITHMS)
print(f"Planned: {PLANNED_INSTANCES} instances, {PLANNED_SOLVER_RUNS} solver runs")


single: 120 available
single: 120 selected; first: C:\Ponpot_Work_Space\math4003-port-yard-optimisation\synthetic_data_generator\singlecommodity_scenarios\F1_system_scale_L\seed_001\F1_system_scale_L_seed_001.json
multi: 120 available
multi: 120 selected; first: C:\Ponpot_Work_Space\math4003-port-yard-optimisation\synthetic_data_generator\multicommodity_scenarios\F1_system_scale_L\seed_001\F1_system_scale_L_seed_001.json
Planned: 240 instances, 480 solver runs


## Run helpers and result storage

Both runtimes below measure elapsed time including model construction and optimisation, excluding file loading and output writing. Solver results, comparisons, and the progress summary are saved after each algorithm completes. A time limit, `MAX_ITERS`, or any missing objective is not labelled an optimal match. Per-solve errors are recorded and the remaining runs continue by default. Set `STOP_ON_ERROR = True` to stop at the first error.

In [4]:
def finite(value):
    if value is None:
        return None
    number = float(value)
    return number if math.isfinite(number) else None

def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, (tuple, list)):
        return [clean_json(v) for v in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def run_milp(module, data):
    model = None
    started = time.perf_counter()
    try:
        model, variables = module.build_model(data, strict_capacity=False)
        model.Params.OutputFlag = 0
        model.Params.MIPGap = BENDERS_TOL
        if DIRECT_TIME_LIMIT_SECONDS is not None:
            model.Params.TimeLimit = DIRECT_TIME_LIMIT_SECONDS
        model.optimize()
        return {
            "status": scenario_runner.STATUS_NAMES.get(model.Status, str(model.Status)),
            "objective": finite(model.ObjVal) if model.SolCount else None,
            "best_bound": finite(model.ObjBound),
            "relative_gap": finite(model.MIPGap) if model.SolCount else None,
            "solution_count": model.SolCount,
            "runtime_seconds": time.perf_counter() - started,
        }
    finally:
        if model is not None:
            model.dispose()

def run_benders(module, data):
    started = time.perf_counter()
    result = module.solve_benders(data, max_iters=BENDERS_MAX_ITERS,
                                  tol=BENDERS_TOL, verbose=VERBOSE_BENDERS)
    row = {name: result.get(name) for name in
           ("status", "iterations", "feasibility_cuts", "optimality_cuts")}
    row.update(objective=finite(result.get("best_objective")),
               runtime_seconds=time.perf_counter() - started)
    return row, result.get("history", [])

FIELDS = ["model", "scenario", "family", "seed", "algorithm", "status", "objective",
          "best_bound", "relative_gap", "solution_count", "iterations", "feasibility_cuts",
          "optimality_cuts", "runtime_seconds", "instance_sha256", "scenario_path", "history_path", "error"]

def save_rows(path, rows, fields):
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR = (PROJECT_ROOT / RESULTS_ROOT / stamp).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=False)

def git_output(*args):
    try:
        return subprocess.check_output(["git", *args], cwd=str(PROJECT_ROOT), text=True,
                                       stderr=subprocess.DEVNULL).strip()
    except (OSError, subprocess.CalledProcessError):
        return None

source_names = ["model_data_loader.py", "monolithic_milp.py", "multicommodity_milp.py",
                "benders_monolithic.py", "benders_multicommodity.py"]
metadata = {
    "started_utc": stamp, "python": platform.python_version(), "gurobi": gp.gurobi.version(),
    "platform": platform.platform(), "processor": platform.processor(), "logical_cpus": os.cpu_count(),
    "git_commit": git_output("rev-parse", "HEAD"), "git_status": git_output("status", "--short"),
    "source_sha256": {name: hashlib.sha256((PROJECT_ROOT / name).read_bytes()).hexdigest() for name in source_names},
    "models": SELECTED_MODELS, "algorithms": SELECTED_ALGORITHMS,
    "instance_paths": PATHS_BY_MODEL, "strict_capacity": False,
    "direct_time_limit_seconds": DIRECT_TIME_LIMIT_SECONDS, "direct_mip_gap": BENDERS_TOL,
    "benders_max_iters": BENDERS_MAX_ITERS, "benders_tolerance": BENDERS_TOL,
    "benders_time_limit_seconds": None, "comparison_rtol": COMPARISON_RTOL,
    "threads_and_solver_seed": "Gurobi defaults; not explicitly overridden",
    "runtime_definition": "elapsed construction and solve time; excludes input loading and output writing",
}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print("Results folder:", RUN_DIR)

def compare_results(rows):
    grouped = {}
    for row in rows:
        grouped.setdefault((row["model"], row["scenario_path"]), {})[row["algorithm"]] = row
    comparisons = []
    for (model, path), pair in grouped.items():
        direct, benders = pair.get("milp", {}), pair.get("benders", {})
        a, b = finite(direct.get("objective")), finite(benders.get("objective"))
        relative_difference = None if a is None or b is None else abs(a - b) / max(1.0, abs(a), abs(b))
        if len(pair) < 2:
            verdict = "NOT_COMPARED"
        elif direct.get("status") == benders.get("status") == "OPTIMAL" and relative_difference is not None:
            verdict = "AGREE_WITHIN_TOLERANCE" if relative_difference <= COMPARISON_RTOL else "REVIEW_DIFFERENCE"
        else:
            verdict = "UNRESOLVED"
        comparisons.append({"model": model, "scenario_path": path,
                            "milp_status": direct.get("status"), "benders_status": benders.get("status"),
                            "milp_objective": a, "benders_objective": b,
                            "relative_objective_difference": relative_difference, "verdict": verdict})
    return comparisons


COMPARISON_FIELDS = ["model", "scenario_path", "milp_status", "benders_status", "milp_objective",
                     "benders_objective", "relative_objective_difference", "verdict"]

def checkpoint(rows):
    comparisons = compare_results(rows)
    save_rows(RUN_DIR / "solver_results.csv", rows, FIELDS)
    save_rows(RUN_DIR / "comparisons.csv", comparisons, COMPARISON_FIELDS)
    status_counts = {}
    for row in rows:
        key = f"{row['model']}/{row['algorithm']}/{row['status']}"
        status_counts[key] = status_counts.get(key, 0) + 1
    verdict_counts = {}
    for row in comparisons:
        verdict_counts[row["verdict"]] = verdict_counts.get(row["verdict"], 0) + 1
    summary = {
        "planned_instances": PLANNED_INSTANCES,
        "planned_solver_runs": PLANNED_SOLVER_RUNS,
        "completed_solver_attempts": len(rows),
        "remaining_solver_attempts": PLANNED_SOLVER_RUNS - len(rows),
        "all_attempts_completed": len(rows) == PLANNED_SOLVER_RUNS,
        "error_count": sum(row["status"] == "ERROR" for row in rows),
        "status_counts": status_counts,
        "comparison_verdict_counts": verdict_counts,
        "all_pairs_agree_within_tolerance": (
            len(SELECTED_ALGORITHMS) == 2 and len(comparisons) == PLANNED_INSTANCES
            and all(row["verdict"] == "AGREE_WITHIN_TOLERANCE" for row in comparisons)
        ),
    }
    (RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    return comparisons, summary


Results folder: C:\Ponpot_Work_Space\math4003-port-yard-optimisation\results\milp_benders\20260907T062321_573096Z


## Run the selected algorithms

In [5]:
results = []
checkpoint(results)
for key, paths in PATHS_BY_MODEL.items():
    direct_module, benders_module = MODULES[key]
    for index, path in enumerate(paths, start=1):
        scenario, family, seed = scenario_runner.describe_path(str(path))
        digest = hashlib.sha256(Path(path).read_bytes()).hexdigest()
        for algorithm in SELECTED_ALGORITHMS:
            print(f"[{key} {index}/{len(paths)}] {scenario}: {algorithm}", flush=True)
            row = {field: None for field in FIELDS}
            row.update(model=key, scenario=scenario, family=family, seed=seed,
                       algorithm=algorithm, scenario_path=str(path), instance_sha256=digest)
            try:
                data = direct_module.load_scenario(path)
                if algorithm == "milp":
                    row.update(run_milp(direct_module, data))
                else:
                    outcome, history = run_benders(benders_module, data)
                    row.update(outcome)
                    history_path = RUN_DIR / f"{key}_{index:04d}_benders_history.json"
                    history_path.write_text(json.dumps(clean_json(history), indent=2, allow_nan=False) + "\n", encoding="utf-8")
                    row["history_path"] = str(history_path)
            except Exception as exc:
                row.update(status="ERROR", error=f"{type(exc).__name__}: {exc}")
            results.append(row)
            checkpoint(results)
            print(" ", row["status"], "objective:", row["objective"], "seconds:", row["runtime_seconds"])
            if row["error"] and STOP_ON_ERROR:
                raise RuntimeError(f"Stopped after saving error to {RUN_DIR / 'solver_results.csv'}: {row['error']}")
print("Completed solver runs:", len(results))

[single 1/120] F1_system_scale_L_seed_001: milp
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2831901
Academic license 2831901 - for non-commercial use only - registered to 28___@curtin.edu.au
  OPTIMAL objective: 12690203.699 seconds: 0.5176494000000034
[single 1/120] F1_system_scale_L_seed_001: benders


KeyboardInterrupt: 

## Compare objectives and review statuses

`AGREE_WITHIN_TOLERANCE` requires both algorithms to report `OPTIMAL` and their relative objective difference to be at most `COMPARISON_RTOL`. The difference is normalised by `max(1, |MILP objective|, |Benders objective|)`. `REVIEW_DIFFERENCE` needs investigation. All other paired outcomes remain `UNRESOLVED`; inspect the statuses and iteration history. Agreement is a numerical regression check, not a proof of algorithmic correctness.

In [ ]:

comparisons, summary = checkpoint(results)
print(json.dumps(summary, indent=2))
try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(results))
    display(pd.DataFrame(comparisons))
except ImportError:
    for row in comparisons:
        print(row)
print("Saved:", RUN_DIR / "solver_results.csv")
print("Saved:", RUN_DIR / "comparisons.csv")
print("Metadata and Benders histories:", RUN_DIR)
print("Progress summary:", RUN_DIR / "summary.json")
needs_review = [row for row in comparisons if row["verdict"] not in ("AGREE_WITHIN_TOLERANCE", "NOT_COMPARED")]
print("Instance comparisons needing review:", len(needs_review))


## Review the full benchmark

The settings already select the full suite. Restart the kernel and Run All. With the default generated datasets, this selects **240 instances and 480 algorithm runs**. Each run gets a new output folder. Open `solver_results.csv` and `comparisons.csv` in Excel, or inspect their tables above. `summary.json` records completed attempts, errors, and agreement counts. Completing all 480 attempts is not the same as all 240 pairs agreeing: check both summary fields.

If the first solve reports a Gurobi license error, select the Python kernel with your working Gurobi installation/license before rerunning. Do not interpret an error, time limit without a solution, or Benders iteration limit as proven infeasibility.

For code-level routing and loader regression tests, run from the repository root:

```powershell
python -m unittest test_milp_canonical_loaders synthetic_data_generator.test_generate_dissertation_scenarios -v
```

This notebook does not run independent instances concurrently or change either solver implementation. Timing results require further controls before manuscript performance claims, particularly if a direct-only time limit is configured.